In [11]:
from opt_targeted_transfers import BinaryGapTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [12]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [13]:
tt = BinaryGapTargetedTransfers(c_bar=2.15, n_regressors=10)

In [14]:
# Nuisance parameter estimation
# Fit conditional improvement regressors for different transfer values
tt.fit(train_dataset, validation_dataset, n_epochs=5)

Fitting conditional binary_gap improvement for transfer size 0.01


 20%|██        | 1/5 [00:00<00:00,  4.74it/s, val loss=1.03]

100%|██████████| 5/5 [00:00<00:00,  6.81it/s, val loss=1.03]


Fitting conditional binary_gap improvement for transfer size 0.2477777777777778


100%|██████████| 5/5 [00:00<00:00,  7.18it/s, val loss=1.04]


...

100%|██████████| 5/5 [00:00<00:00,  8.40it/s, val loss=1.1]


Fitting conditional binary_gap improvement for transfer size 1.9122222222222223


100%|██████████| 5/5 [00:00<00:00,  7.31it/s, val loss=1.1]


Fitting conditional binary_gap improvement for transfer size 2.15


100%|██████████| 5/5 [00:00<00:00, 24.70it/s, val loss=1.1]


In [31]:
# Precomputation for policy optimization step.
import numpy as np
budgets = np.linspace(0.05, 2.15, 50)
tt.optimize_transfers_for_budget_grid(test_covariate_dataset, budgets=budgets)

In [28]:
# Set budget and run policy optimization step for that budget.
# Policy optimization step returns transfer amount for each unit in the test set.
# Note that budget must lie in the set of budgets used in the precomputation step.
tt.set_budget(budget=budgets[2])
assignments = tt.run_opt(test_covariate_dataset)

In [29]:
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.2767705373738549,
 'post_transfer_poverty_rate': 0.45526918557140383,
 'policy_cost_per_capita': 0.34989273396852233,
 'budget': 0.35,
 'policy_type': 'binary_gap',
 'd': 2}

In [18]:
# Can try a different budget without redoing the fit step and precomputation step.
# Note that budget must lie in the set of budgets used in the precomputation step.
# Setting the budget will clear assignments attribute.
tt.set_budget(1.5)
tt.run_opt(test_covariate_dataset)
res = tt.evaluate(test_dataset)
res


{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.004788081492851036,
 'post_transfer_poverty_rate': 0.039397006670417595,
 'policy_cost_per_capita': 1.4366666666666712,
 'budget': 1.5,
 'policy_type': 'binary_gap',
 'd': 2}